In [7]:
from pyspark.sql import SparkSession
import numpy as np
import pandas as pd


spark = (
    SparkSession.builder.appName("MAST30034 Project 2")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

In [8]:
# simple cleaning checks and removals

all_rentals = pd.read_csv('rea_data/domain/Data/vic_rentals_all.csv')

print(f'Total listings: {all_rentals.shape}')

print(f'No Rent: {all_rentals[(all_rentals["weekly_rent"] <= 0) | (all_rentals["weekly_rent"].isna())].shape}')
rental_cleaning = all_rentals[(all_rentals["weekly_rent"] > 0) & (~all_rentals["weekly_rent"].isna())]

print(f'No Bond: {rental_cleaning[(rental_cleaning["bond"] <= 0) | (rental_cleaning["bond"].isna())].shape}')

print(f'No Suburb: {rental_cleaning[rental_cleaning["suburb"].isna()].shape}')
rental_cleaning["suburb"] = rental_cleaning["suburb"].str.lower()

print(f'No Address: {rental_cleaning[rental_cleaning["address"].isna()].shape}')

print(f'No Bedrooms: {rental_cleaning[(rental_cleaning["bedrooms"] <= 0) | (rental_cleaning["bedrooms"].isna())].shape}')

print(f'No Bathrooms: {rental_cleaning[(rental_cleaning["bathrooms"] <= 0) | (rental_cleaning["bathrooms"].isna())].shape}')

print(f'No Car Spaces: {rental_cleaning[(rental_cleaning["carspaces"] <= 0) | (rental_cleaning["carspaces"].isna())].shape}')

print(f'Types of Listings: {rental_cleaning["property_type"].unique()}')

print(f'Same Lisitngs: {rental_cleaning[rental_cleaning.duplicated(subset=["domain_page_id"], keep=False)].shape}')
print(f'Incorrect ID: {rental_cleaning[rental_cleaning.duplicated(subset=["domain_page_id"], keep=False)]["domain_page_id"].unique()}')

rental_cleaning = rental_cleaning.dropna(subset=["address"])
rental_cleaning = rental_cleaning.dropna(subset=["bedrooms"])
rental_cleaning = rental_cleaning.dropna(subset=["bathrooms"])
rental_cleaning = rental_cleaning.drop_duplicates(subset="domain_page_id", keep=False)

Total listings: (12717, 30)
No Rent: (296, 30)
No Bond: (747, 30)
No Suburb: (0, 30)
No Address: (89, 30)
No Bedrooms: (108, 30)
No Bathrooms: (36, 30)
No Car Spaces: (1745, 30)
Types of Listings: ['Apartment / Unit / Flat' 'Townhouse' 'House' 'Studio' 'Duplex' 'Villa'
 'Block of Units' 'Farm' 'New House & Land' 'Vacant land' 'Semi-Detached'
 'Acreage / Semi-Rural' 'Car Space' 'Terrace'
 'New Apartments / Off the Plan']
Same Lisitngs: (3, 30)
Incorrect ID: ['search results - rent']


/tmp/ipykernel_1691/2036684324.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rental_cleaning["suburb"] = rental_cleaning["suburb"].str.lower()


In [9]:
# Investiagte the missing bonds

suburb_counts = pd.read_csv('rea_data/domain/Data/suburb_summary.csv')
suburb_counts['suburb'] = suburb_counts['suburb'].str.lower()

missing_counts = (
    rental_cleaning[
        (rental_cleaning["bond"].isna()) | (rental_cleaning["bond"] <= 0)]
    .groupby("suburb")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="missing_bonds")
)

missing_prop = suburb_counts.merge(
    missing_counts,
    left_on="suburb",
    right_on="suburb",
    how="left"
)

missing_prop['missing_percent'] = 100 * missing_prop['missing_bonds'] / missing_prop['listing_count']
print(missing_prop.sort_values(by='missing_percent', ascending=False).head(30))

original_stats = rental_cleaning["weekly_rent"].describe()

cleaned = rental_cleaning[~((rental_cleaning["bond"].isna()) | (rental_cleaning["bond"] <= 0))]
cleaned_stats = cleaned["weekly_rent"].describe()

print("Original Weekly Rent Stats:\n", original_stats)
print("\nAfter Removing Missing/Invalid Bond Rows:\n", cleaned_stats)

#rental_cleaning = rental_cleaning[~((rental_cleaning["bond"].isna()) | (rental_cleaning["bond"] <= 0))]

               suburb  postcode  listing_count  missing_bonds  missing_percent
618         mandurang      3551              1            1.0       100.000000
449               yea      3717              2            1.0        50.000000
474         penshurst      3289              2            1.0        50.000000
356         mansfield      3722              6            3.0        50.000000
455            eildon      3713              2            1.0        50.000000
321            cobram      3644              7            3.0        42.857143
249       tullamarine      3043             12            5.0        41.666667
379        myrtleford      3737              5            2.0        40.000000
168  avondale heights      3034             21            8.0        38.095238
261         bell park      3215             11            4.0        36.363636
258        travancore      3032             11            4.0        36.363636
444             nyora      3987              3      

In [10]:
# Investiagte rent distribution
top_expensive = rental_cleaning.sort_values(by="weekly_rent", ascending=False)

print("Top 10 Most Expensive Properties:")
print(top_expensive[["listing_id", "suburb", "address", "weekly_rent"]].head(20))

rental_cleaning = rental_cleaning[rental_cleaning["weekly_rent"] < 6000]

Top 10 Most Expensive Properties:
       listing_id           suburb                     address  weekly_rent
2586     17697966          berwick          83 Golf Links Road     808500.0
3641     17750082          rosebud            4 Wilfred Street     595000.0
1929     17747509          morwell           1/28 Elgin Street     315000.0
12294    17661725       camberwell          54 Fairmont Avenue       5866.0
10358    17722803           toorak             2 Lisbuoy Court       5750.0
10768    17694334      south yarra             1 Fairlie Court       4500.0
8776     17744678         hawthorn           94 Illawarra Road       4500.0
12677    17750966         brighton          74 Champion Street       4000.0
1236     17460975  south melbourne        901/161 Eastern Road       4000.0
1625     14253235   port melbourne                 1 Tarver St       3900.0
3704     17721041         cremorne               13 Balmain St       3850.0
10707    17666152      south yarra          16 St Mart

In [11]:
# bathrooms and bedrooms dist
inconsistent = rental_cleaning[(rental_cleaning["bathrooms"] > rental_cleaning["bedrooms"] + 2)]
print(inconsistent[['listing_id', 'bathrooms']])
rental_cleaning = rental_cleaning[~rental_cleaning["listing_id"].isin(inconsistent["listing_id"])]


print(rental_cleaning[['listing_id', 'bedrooms']].sort_values('bedrooms', ascending=False).head(5))
rental_cleaning = rental_cleaning[rental_cleaning['bedrooms'] < 11]

      listing_id  bathrooms
1085    17601834        7.0
1279    17030693        9.0
6909    17753508       12.0
6948    16916036        9.0
7494    17211921        6.0
      listing_id  bedrooms
5362    15800260      50.0
2133    17748261      11.0
2011    17737763       9.0
3838    17492148       9.0
5146    17455105       9.0


In [12]:
# save the final cleaned data
print(f'Final Shape: {rental_cleaning.shape}')
rental_cleaning.to_csv('rea_data/domain/Data/vic_rentals_all_cleaned.csv', index=False)

Final Shape: (12210, 30)
